# Labb #2 – More Statistical Inference and Data Analytics
**Data Science & Machine Learning: 7,5 hp**  
**Gruppnummer: 15**

---
# Uppgift 8 – ANOVA: Analys av globala supporttider (ANOVA vs. Kruskal-Wallis)

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# --- STEG 1: Generera gruppspecifikt dataset ---
gruppnummer = 15
rng = np.random.default_rng(gruppnummer)

center = ['Sverige', 'Indien', 'USA']
n_per_group = 50

# Grupp 15 är udda => normalfördelat
dist_type = "skewed" if gruppnummer % 2 == 0 else "normal"

data_list = []
for i, c in enumerate(center):
    loc = 10 + (gruppnummer % 3) + i
    if dist_type == "normal":
        values = rng.normal(loc=loc, scale=2.5, size=n_per_group)
    else:
        values = rng.lognormal(mean=np.log(loc), sigma=0.5, size=n_per_group)
    temp_df = pd.DataFrame({'Center': c, 'Svarstid': values})
    data_list.append(temp_df)

df = pd.concat(data_list).reset_index(drop=True)
print(f"Data genererad för grupp {gruppnummer} (Typ: {dist_type})")
print(df.groupby('Center')['Svarstid'].describe().round(2))

ModuleNotFoundError: No module named 'statsmodels'

## Deluppgift 1: Explorativ Dataanalys (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Density / Histogram per center
for c in center:
    grp = df[df['Center'] == c]['Svarstid']
    axes[0].hist(grp, bins=15, alpha=0.5, label=c, edgecolor='black')
axes[0].set_title('Histogram – Svarstider per supportcenter')
axes[0].set_xlabel('Svarstid (timmar)')
axes[0].set_ylabel('Antal')
axes[0].legend()

# Boxplot
sns.boxplot(data=df, x='Center', y='Svarstid', ax=axes[1], palette='Set2')
axes[1].set_title('Boxplot – Svarstider per supportcenter')
axes[1].set_xlabel('Supportcenter')
axes[1].set_ylabel('Svarstid (timmar)')

plt.tight_layout()
plt.show()

print("Visuell analys:")
print("- Histogrammen ser approximativt normalfördelade ut för alla tre center.")
print("- Boxplottarna visar liknande spridning (varians) för de tre grupperna.")
print("- USA har ett något högre medelvärde, Sverige lägst.")

## Deluppgift 2: Test av antaganden – Shapiro-Wilk normalitetstest

In [ ]:
print("Shapiro-Wilk normalitetstest per supportcenter:")
print("-" * 50)
alla_normal = True
for c in center:
    grp = df[df['Center'] == c]['Svarstid']
    stat, p = stats.shapiro(grp)
    normalfordeld = "JA" if p > 0.05 else "NEJ"
    print(f"{c}: W={stat:.4f}, p={p:.4f} => Normalfördelad? {normalfordeld}")
    if p <= 0.05:
        alla_normal = False

print()
if alla_normal:
    print("Slutsats: Alla grupper är normalfördelade (p > 0.05 för alla).")
    print("=> Vi går vidare med One-way ANOVA.")
else:
    print("Slutsats: Minst en grupp avviker från normalfördelning.")
    print("=> Vi provar logtransformation och/eller Kruskal-Wallis.")

## Deluppgift 3: Hypotesprövning – One-way ANOVA

In [ ]:
print("Hypoteser:")
print("H0: Det finns INGEN signifikant skillnad i medelsvarstid mellan de tre supportcentren.")
print("    μ_Sverige = μ_Indien = μ_USA")
print("Ha: Minst ett center har en signifikant annorlunda medelsvarstid.")
print()

# Extrahera grupper
sverige = df[df['Center'] == 'Sverige']['Svarstid']
indien  = df[df['Center'] == 'Indien']['Svarstid']
usa     = df[df['Center'] == 'USA']['Svarstid']

# One-way ANOVA
f_stat, p_anova = stats.f_oneway(sverige, indien, usa)

print(f"One-way ANOVA:")
print(f"  F-statistik : {f_stat:.4f}")
print(f"  p-värde     : {p_anova:.4f}")
print()

alpha = 0.05
if p_anova < alpha:
    print(f"Slutsats: p={p_anova:.4f} < alpha={alpha} => Vi FÖRKASTAR H0.")
    print("Det finns en statistiskt signifikant skillnad i svarstider mellan minst två center.")
else:
    print(f"Slutsats: p={p_anova:.4f} >= alpha={alpha} => Vi KAN INTE förkasta H0.")
    print("Det finns ingen statistiskt signifikant skillnad i svarstider mellan centren.")

## Deluppgift 4: Post-hoc test – Tukey's HSD

In [ ]:
# Post-hoc körs endast om ANOVA var signifikant
alpha = 0.05
if p_anova < alpha:
    print("ANOVA var signifikant => Kör Tukey's HSD post-hoc test")
    print()
    tukey = pairwise_tukeyhsd(endog=df['Svarstid'], groups=df['Center'], alpha=0.05)
    print(tukey)
    print()
    print("Tolkning: Rader med 'True' i kolumnen 'reject' indikerar signifikanta skillnader.")
else:
    print("ANOVA var INTE signifikant (p >= 0.05).")
    print("Post-hoc test behövs ej – vi kan inte förkasta H0.")
    print("Slutsats: Det finns ingen statistiskt signifikant skillnad mellan centren.")

---
# Uppgift 9 – Chi²-test: Analys av Kundnöjdhet

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# --- STEG 1: Generera gruppspecifikt dataset ---
gruppnummer = 15
rng = np.random.default_rng(gruppnummer)

kategorier = ['Inte nöjd', 'Neutral', 'Nöjd', 'Mycket nöjd']
p_expected = [0.10, 0.20, 0.40, 0.30]

offset = (gruppnummer % 10 - 5) * 0.02
p_group = [
    max(0.05, p_expected[0] + offset),
    max(0.05, p_expected[1] - offset),
    p_expected[2],
    1.0 - (max(0.05, p_expected[0] + offset) + max(0.05, p_expected[1] - offset) + p_expected[2])
]

antal_svar = 200
kundsvar = rng.choice(kategorier, size=antal_svar, p=p_group)
data = pd.Series(kundsvar)

print(f"Dataset för grupp {gruppnummer} genererat.")
print(data.value_counts())

## Hypotesformulering

In [ ]:
print("Hypoteser:")
print("H0: Företagets kundnöjdhetsfördelning följer branschstandarden (10%, 20%, 40%, 30%).")
print("Ha: Företagets kundnöjdhetsfördelning AVVIKER från branschstandarden.")

## Beräkning av förväntade frekvenser

In [ ]:
# Observerade frekvenser i rätt ordning
observerade = [data.value_counts().get(k, 0) for k in kategorier]

# Förväntade frekvenser baserat på branschstandard och n=200
forväntade = [p * antal_svar for p in p_expected]

jämförelse = pd.DataFrame({
    'Kategori': kategorier,
    'Observerad': observerade,
    'Förväntad (bransch)': forväntade
})
print(jämförelse.to_string(index=False))

## Genomförande – Chi²-test

In [ ]:
chi2_stat, p_chi2 = stats.chisquare(f_obs=observerade, f_exp=forväntade)

print(f"Chi²-statistik : {chi2_stat:.4f}")
print(f"p-värde        : {p_chi2:.4f}")
print()

alpha = 0.05
if p_chi2 < alpha:
    print(f"Slutsats: p={p_chi2:.4f} < alpha={alpha} => Vi FÖRKASTAR H0.")
    print("Företagets kundnöjdhetsfördelning skiljer sig signifikant från branschstandarden.")
else:
    print(f"Slutsats: p={p_chi2:.4f} >= alpha={alpha} => Vi KAN INTE förkasta H0.")
    print("Det finns ingen statistiskt signifikant skillnad mot branschstandarden.")
    print("E-handelsbolagets kundnöjdhet verkar följa branschstandarden.")

## Visualisering – Observerade vs. Förväntade frekvenser

In [ ]:
x = np.arange(len(kategorier))
bredd = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - bredd/2, observerade, bredd, label='Observerad', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + bredd/2, forväntade,  bredd, label='Förväntad (bransch)', color='coral', edgecolor='black')

ax.set_xlabel('Nöjdhetskategori')
ax.set_ylabel('Antal svar')
ax.set_title(f'Kundnöjdhet – Grupp {gruppnummer}: Observerade vs. Förväntade frekvenser\n'
             f'Chi²={chi2_stat:.2f}, p={p_chi2:.4f}')
ax.set_xticks(x)
ax.set_xticklabels(kategorier)
ax.legend()

# Lägg till värden ovanpå staplarna
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---
# Uppgift 10 – Diagnostisk precision med Fishers exakta test

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# --- STEG 1: Generera gruppspecifikt dataset ---
gruppnummer = 15
rng = np.random.default_rng(gruppnummer)

ai_success_rate     = 0.7 + (gruppnummer % 4) * 0.05   # = 0.85
manual_success_rate = 0.4 + (gruppnummer % 3) * 0.05   # = 0.40

ai_group     = rng.binomial(n=1, p=ai_success_rate,     size=12)
manual_group = rng.binomial(n=1, p=manual_success_rate, size=12)

ai_korrekt     = np.sum(ai_group)
ai_fel         = 12 - ai_korrekt
manual_korrekt = np.sum(manual_group)
manual_fel     = 12 - manual_korrekt

tabell = np.array([[ai_korrekt, ai_fel],
                   [manual_korrekt, manual_fel]])

data_df = pd.DataFrame(tabell,
                       index=['AI-verktyg', 'Manuell metod'],
                       columns=['Korrekt diagnos', 'Felaktig diagnos'])

print(f"Kontingenstabell för grupp {gruppnummer}:")
print(data_df)

## Deluppgift 1: Teoretisk motivering – Varför Fishers exakta test?

Vi använder **Fishers exakta test** istället för Chi²-test av följande skäl:

Chi²-testet är en approximation som kräver att de **förväntade cellfrekvenserna är minst 5** i alla celler. Med totalt 24 patienter fördelade på en 2×2-tabell riskerar flera celler att ha förväntade frekvenser under 5, vilket gör Chi²-approximationen opålitlig.

Fishers exakta test beräknar exakt sannolikhet utan approximation, vilket gör det idealt för **små stickprov**.

In [ ]:
# Visa förväntade frekvenser för att motivera valet
n = tabell.sum()
rad_summor = tabell.sum(axis=1)
kol_summor = tabell.sum(axis=0)
print("Förväntade frekvenser (rad_summa × kol_summa / n):")
for i, rad in enumerate(['AI-verktyg', 'Manuell metod']):
    for j, kol in enumerate(['Korrekt diagnos', 'Felaktig diagnos']):
        förväntat = (rad_summor[i] * kol_summor[j]) / n
        print(f"  {rad} / {kol}: {förväntat:.2f}")
print()
print("Celler med förväntad frekvens < 5 gör Chi²-testet opålitligt => Fishers exakta test används.")

## Deluppgift 2: Hypotesformulering

In [ ]:
print("Hypoteser:")
print("H0: Det finns INGET samband mellan metod (AI vs. manuell) och diagnosprecision.")
print("    AI-verktyget är inte bättre än manuell metod.")
print()
print("Ha: Det finns ETT SAMBAND mellan metod och diagnosprecision.")
print("    AI-verktyget ger en annorlunda (bättre) precisioner än manuell metod.")

## Deluppgift 3: Genomförande – Fishers exakta test

In [ ]:
odds_ratio, p_fisher = stats.fisher_exact(tabell)

print(f"Fishers exakta test – resultat:")
print(f"  Odds Ratio : {odds_ratio:.4f}")
print(f"  p-värde    : {p_fisher:.4f}")

## Deluppgift 5: Analys och tolkning

In [ ]:
alpha = 0.05
print(f"Signifikansnivå: alpha = {alpha}")
print(f"p-värde        : {p_fisher:.4f}")
print()

if p_fisher < alpha:
    print(f"Slutsats: p={p_fisher:.4f} < alpha={alpha} => Vi FÖRKASTAR H0.")
    print("Det finns ett statistiskt signifikant samband mellan metod och diagnosprecision.")
    print("AI-verktyget ger signifikant bättre diagnostisk precision än manuell metod.")
else:
    print(f"Slutsats: p={p_fisher:.4f} >= alpha={alpha} => Vi KAN INTE förkasta H0.")
    print("Det finns inget statistiskt signifikant samband mellan metod och diagnosprecision.")

print()
print(f"Odds Ratio-tolkning:")
print(f"  OR = {odds_ratio:.2f} innebär att oddsen att få en korrekt diagnos är")
print(f"  {odds_ratio:.1f} gånger högre med AI-verktyget jämfört med manuell metod.")
if odds_ratio > 1:
    print(f"  Detta tyder på att AI-verktyget presterar avsevärt bättre i detta dataset.")